first seeing differences in names between both tables

Recurrence was used as metric because has 158 events out of 670 patients 
Metastasis events: 35


In [ ]:
import pandas as pd
import numpy as np

# Load both files
deconv = pd.read_csv('/path/to/data/cell_type_deconvolution_PPCG_TME.csv')
metadata = pd.read_csv('/path/to/data/metadata_common_PPCG.tsv', sep='\t')

# Check what they look like
print("Deconv shape:", deconv.shape)
print("Deconv columns:", deconv.columns.tolist())
print("Deconv head:")
print(deconv.head(3))

print("\nMetadata shape:", metadata.shape)
print("Metadata columns:", metadata.columns.tolist())
print("Metadata head:")
print(metadata.head(3))

In [ ]:
# Extract patient ID from assay ID
# PPCG0392e_RNAseq -> PPCG0392
deconv['ppcg_id'] = deconv['PPCG_RNA_Assay_ID'].str.extract(r'(PPCG\d+)')

print("Example extraction:")
print(deconv[['PPCG_RNA_Assay_ID', 'ppcg_id']].head(10))

# Check how many unique patients in deconv
print(f"\nUnique patients in deconv: {deconv['ppcg_id'].nunique()}")
print(f"Unique patients in metadata: {metadata['ppcg_id'].nunique()}")

# Check overlap
overlap = set(deconv['ppcg_id']) & set(metadata['ppcg_id'])
print(f"Overlapping patients: {len(overlap)}")

# Check how many samples per patient (some have multiple biopsies)
print("\nSamples per patient (top 10):")
print(deconv['ppcg_id'].value_counts().head(10))

In [ ]:
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# ── Cell type columns ─────────────────────────────────────────────────
cell_type_cols = ['epithelial', 'plasma', 'endothelial', 't_cd8', 't_cd4',
                  'b', 'macrophage', 'mast', 'monocyte', 'natural_killer',
                  't_regulatory', 'dendritic', 'fibroblast', 'smooth_muscle']

# ── Replace 999 with NaN in metadata ─────────────────────────────────
clinical_cols = ['gleason_grade_group', 'path_gleason_sum', 'damico_risk',
                 'relapse_ind', 'donor_relapse_interval',
                 'donor_interval_of_last_followup',
                 'death_ind', 'age_at_tumour_collection',
                 'psa_at_tumour_collection']

metadata_clean = metadata.copy()
for col in clinical_cols:
    if col in metadata_clean.columns:
        metadata_clean[col] = metadata_clean[col].replace(999, np.nan)

# ── Aggregate deconv to patient level ────────────────────────────────
deconv_patient = deconv.groupby('ppcg_id')[cell_type_cols].mean()

# ── Merge with metadata ───────────────────────────────────────────────
merged = deconv_patient.merge(
    metadata_clean.set_index('ppcg_id')[clinical_cols],
    left_index=True, right_index=True,
    how='inner'
)
merged.to_csv('PPCG_patient_level.csv')
print(f"Patient-level matrix: {merged.shape}")
print(f"Relapse events: {merged['relapse_ind'].sum():.0f} / {merged['relapse_ind'].notna().sum():.0f} patients with data")
print(f"Missing follow-up time: {merged['donor_interval_of_last_followup'].isna().sum()}")
print(f"Missing relapse time: {merged['donor_relapse_interval'].isna().sum()}")

In [ ]:
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── Build survival dataframe ──────────────────────────────────────────
# For each patient:
# - if relapsed: time = donor_relapse_interval, event = 1
# - if not relapsed: time = donor_interval_of_last_followup, event = 0

surv = merged.copy()

surv['time'] = np.where(
    surv['relapse_ind'] == 1,
    surv['donor_relapse_interval'],
    surv['donor_interval_of_last_followup']
)
surv['event'] = surv['relapse_ind'].fillna(0).astype(int)

# Drop rows with missing time
surv = surv.dropna(subset=['time'])
surv = surv[surv['time'] > 0]

print(f"Patients in survival analysis: {len(surv)}")
print(f"Events (relapses): {surv['event'].sum()}")
print(f"Median follow-up: {surv['time'].median():.1f} days")

# ── Cell types to test — exclude those with near-zero median ─────────
test_cols = ['epithelial', 'smooth_muscle', 'fibroblast',
             'endothelial', 'macrophage']

# ── Kaplan-Meier curves — median split per cell type ─────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, ct in enumerate(test_cols):
    ax = axes[i]
    median_val = surv[ct].median()
    high = surv[surv[ct] >= median_val]
    low  = surv[surv[ct] <  median_val]

    # Log-rank test
    result = logrank_test(
        high['time'], low['time'],
        event_observed_A=high['event'],
        event_observed_B=low['event']
    )

    kmf_high = KaplanMeierFitter()
    kmf_low  = KaplanMeierFitter()

    kmf_high.fit(high['time'], high['event'], label=f'High (n={len(high)})')
    kmf_low.fit(low['time'],   low['event'],  label=f'Low (n={len(low)})')

    kmf_high.plot_survival_function(ax=ax, ci_show=True, color='tomato')
    kmf_low.plot_survival_function(ax=ax, ci_show=True, color='steelblue')

    ax.set_title(f'{ct}\np={result.p_value:.4f}', fontsize=11)
    ax.set_xlabel('Time (days)')
    ax.set_ylabel('Relapse-free survival')
    ax.legend(fontsize=9)

# Hide unused subplot
axes[-1].set_visible(False)

plt.suptitle(
    'Relapse-free survival by cell type proportion\n(median split, PPCG cohort)',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig('KM_curves_cell_types.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved KM curves')

# ── Cox PH multivariate model ─────────────────────────────────────────
cox_cols = test_cols + ['gleason_grade_group', 'age_at_tumour_collection']

cox_data = surv[cox_cols + ['time', 'event']].dropna()
print(f"\nCox PH patients (after dropping NaN): {len(cox_data)}")

cph = CoxPHFitter()
cph.fit(cox_data, duration_col='time', event_col='event')
cph.print_summary()

# Save forest plot
fig, ax = plt.subplots(figsize=(8, 6))
cph.plot(ax=ax)
ax.set_title(
    'Cox PH — cell type proportions + Gleason grade\n(outcome: relapse-free survival)',
    fontsize=11
)
ax.axvline(0, color='black', linestyle='--', linewidth=0.8)
plt.tight_layout()
plt.savefig('CoxPH_cell_types.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved Cox PH forest plot')

there might be collinearity between gleason grade and cell types, so we run cox models separately

In [ ]:
from lifelines import CoxPHFitter
import pandas as pd
import numpy as np

results = []

for ct in test_cols:
    cox_data = surv[[ct, 'time', 'event']].dropna()
    cph = CoxPHFitter()
    cph.fit(cox_data, duration_col='time', event_col='event')
    
    summary = cph.summary
    results.append({
        'cell_type': ct,
        'coef': summary.loc[ct, 'coef'],
        'HR': summary.loc[ct, 'exp(coef)'],
        'HR_lower': summary.loc[ct, 'exp(coef) lower 95%'],
        'HR_upper': summary.loc[ct, 'exp(coef) upper 95%'],
        'p': summary.loc[ct, 'p']
    })

results_df = pd.DataFrame(results)
results_df['padj'] = results_df['p'] * len(results_df)  # Bonferroni
results_df['padj'] = results_df['padj'].clip(upper=1.0)
print(results_df.round(4).to_string())
results_df.to_csv('CoxPH_univariate_cell_types.csv', index=False)

In [ ]:
print("Cell type proportion distributions:")
print(surv[test_cols].describe().round(4))
print("\nStandard deviations:")
print(surv[test_cols].std().round(4))

right approach here is rank-based normalization rather than z-scoring, since these distributions are clearly non-normal. for every 1 SD increase in the ranked cell type proportion, how does the instantaneous risk of relapse change


In [ ]:
from scipy.stats import rankdata

results = []
for ct in test_cols:
    cox_data = surv[[ct, 'time', 'event']].dropna().copy()
    
    # Rank-based inverse normal transformation
    # Converts to standard normal regardless of original distribution
    n = len(cox_data)
    ranks = rankdata(cox_data[ct])
    # Blom's formula
    cox_data[f'{ct}_ranked'] = (ranks - 0.375) / (n + 0.25)
    # Convert to z-scores via inverse normal
    from scipy.stats import norm
    cox_data[f'{ct}_ranked'] = norm.ppf(cox_data[f'{ct}_ranked'])
    
    cph = CoxPHFitter()
    cph.fit(cox_data[[f'{ct}_ranked', 'time', 'event']],
            duration_col='time', event_col='event')
    
    summary = cph.summary
    results.append({
        'cell_type': ct,
        'HR (per 1 SD rank)': summary.loc[f'{ct}_ranked', 'exp(coef)'],
        'HR_lower': summary.loc[f'{ct}_ranked', 'exp(coef) lower 95%'],
        'HR_upper': summary.loc[f'{ct}_ranked', 'exp(coef) upper 95%'],
        'p_val': summary.loc[f'{ct}_ranked', 'p']
    })

results_df = pd.DataFrame(results)
results_df['padj'] = (results_df['p_val'] * len(results_df)).clip(upper=1.0)
print("=== Univariate Cox PH (rank-based inverse normal transformation) ===")
print(results_df.round(4).to_string(index=False))
results_df.to_csv('CoxPH_univariate_ranked.csv', index=False)

forest plot of HRs with confidence intervals:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ── Data ──────────────────────────────────────────────────────────────
results_df = results_df.sort_values('HR (per 1 SD rank)', ascending=True)

labels = results_df['cell_type'].tolist()
hrs    = results_df['HR (per 1 SD rank)'].tolist()
lower  = results_df['HR_lower'].tolist()
upper  = results_df['HR_upper'].tolist()
padj   = results_df['padj'].tolist()

# Confidence interval errors for errorbar
xerr_low  = [hr - lo for hr, lo in zip(hrs, lower)]
xerr_high = [hi - hr for hr, hi in zip(hrs, upper)]

# ── Colours: red = risk, blue = protective, grey = NS ─────────────────
colors = []
for hr, p in zip(hrs, padj):
    if p >= 0.05:
        colors.append('#999999')
    elif hr > 1:
        colors.append('#d62728')
    else:
        colors.append('#1f77b4')

# ── Significance stars ────────────────────────────────────────────────
def sig_label(p):
    if p < 0.001:  return '***'
    elif p < 0.01: return '**'
    elif p < 0.05: return '*'
    else:          return 'ns'

sig_labels = [sig_label(p) for p in padj]

# ── Plot ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

y_pos = np.arange(len(labels))

ax.errorbar(
    hrs, y_pos,
    xerr=[xerr_low, xerr_high],
    fmt='o',
    color='black',
    ecolor='black',
    elinewidth=1.5,
    capsize=4,
    capthick=1.5,
    markersize=0  # hide default marker; use scatter below
)

# Coloured markers
for i, (hr, y, c) in enumerate(zip(hrs, y_pos, colors)):
    ax.scatter(hr, y, color=c, s=80, zorder=5)

# Significance labels
for i, (hr, hi, y, sl) in enumerate(zip(hrs, upper, y_pos, sig_labels)):
    ax.text(hi + 0.02, y, f'  {sl}', va='center', fontsize=10)

# Reference line at HR=1
ax.axvline(1.0, color='black', linestyle='--', linewidth=1, alpha=0.6)

# Axis labels
ax.set_yticks(y_pos)
ax.set_yticklabels([l.replace('_', ' ').title() for l in labels], fontsize=11)
ax.set_xlabel('Hazard Ratio (per 1 SD increase in ranked proportion)\nwith 95% confidence interval', fontsize=10)
ax.set_title(
    'Univariate Cox PH — cell type proportions vs relapse-free survival\n'
    'PPCG cohort (n=670, 158 events) | Bonferroni-adjusted',
    fontsize=11, fontweight='bold'
)

# Annotation box
ax.text(
    0.98, 0.02,
    '*** padj<0.001   ** padj<0.01   * padj<0.05   ns not significant',
    transform=ax.transAxes,
    fontsize=8, ha='right', va='bottom',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='grey', alpha=0.8)
)

# Shade risk vs protective regions

ax.axvspan(ax.get_xlim()[0], 1.0, alpha=0.04, color='steelblue')
ax.axvspan(1.0, ax.get_xlim()[1], alpha=0.04, color='tomato')
ax.text(0.97, 0.55, 'Risk →', transform=ax.transAxes,
        fontsize=9, color='tomato', ha='right', alpha=0.7)
ax.text(0.03, 0.55, '← Protective', transform=ax.transAxes,
        fontsize=9, color='steelblue', ha='left', alpha=0.7)

plt.tight_layout()
plt.savefig('ForestPlot_CoxPH_cell_types.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved forest plot')

changing km curve to show the bonferroni padj instead of raw pvalue

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, ct in enumerate(test_cols):
    ax = axes[i]
    median_val = surv[ct].median()
    high = surv[surv[ct] >= median_val]
    low  = surv[surv[ct] <  median_val]

    # Log-rank test
    result = logrank_test(
        high['time'], low['time'],
        event_observed_A=high['event'],
        event_observed_B=low['event']
    )
    
    # Bonferroni-adjusted p-value
    padj = min(result.p_value * len(test_cols), 1.0)

    kmf_high = KaplanMeierFitter()
    kmf_low  = KaplanMeierFitter()

    kmf_high.fit(high['time'], high['event'], label=f'High (n={len(high)})')
    kmf_low.fit(low['time'],   low['event'],  label=f'Low (n={len(low)})')

    # Consistent colours: steelblue=high, tomato=low
    kmf_high.plot_survival_function(ax=ax, ci_show=True, color='steelblue')
    kmf_low.plot_survival_function(ax=ax, ci_show=True, color='tomato')

    ax.set_title(f'{ct}\npadj={padj:.4f}', fontsize=11)
    ax.set_xlabel('Time (days)')
    ax.set_ylabel('Relapse-free survival')
    ax.legend(fontsize=9)

axes[-1].set_visible(False)

plt.suptitle(
    'Relapse-free survival by cell type proportion\n(median split, PPCG cohort | Bonferroni-adjusted)',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig('KM_curves_cell_types.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved KM curves')

multivariate cox

In [ ]:
# Rank-transform all cell types
cox_multi = surv[test_cols + ['time', 'event', 'gleason_grade_group']].dropna().copy()

for ct in test_cols:
    ranks = rankdata(cox_multi[ct])
    n = len(cox_multi)
    cox_multi[f'{ct}_ranked'] = norm.ppf((ranks - 0.375) / (n + 0.25))

ranked_cols = [f'{ct}_ranked' for ct in test_cols]
cph_multi = CoxPHFitter()
cph_multi.fit(
    cox_multi[ranked_cols + ['gleason_grade_group', 'time', 'event']],
    duration_col='time',
    event_col='event'
)
cph_multi.print_summary()

fibroblast proportion predicts relapse independently of Gleason grade: even when two patients have the same Gleason grade, the one with more fibroblasts in their tumour still has a higher risk of relapse

Univariate Cox
Tests each cell type in isolation. Answers: "does this cell type predict relapse on its own?" This is what the forest plot shows because it is the most interpretable and directly answers your biological question for each cell type independently. This is what goes in the thesis as the main result.
Multivariate Cox
Tests all cell types simultaneously alongside Gleason grade. Answers: "does this cell type predict relapse independently of everything else?" We ran this to check whether the cell type associations were just reflecting Gleason grade (which they mostly were, due to collinearity) or whether any had truly independent information. Only fibroblast survived multivariate adjustment